# 2 · Translation — Model Training

1. Load preprocessed artifacts
2. Build `tf.data.Dataset` pipelines
3. Instantiate encoder + decoder
4. Run custom teacher-forced training loop
5. Save encoder and decoder weights → `models/translation/`
6. Plot training loss curve

## 0 · Configuration

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ── USER SETTINGS ─────────────────────────────────────────────────────────────
LANGUAGE       = 'french'   # 'french' | 'spanish' | 'german' | 'hindi'
EPOCHS         = 20         # Training epochs (EarlyStopping will cut short if needed)
BATCH_SIZE     = 64
LEARNING_RATE  = 1e-3

# ── RESUME SETTINGS ───────────────────────────────────────────────────────
RESUME         = True     # Set True to load saved weights and continue
START_EPOCH    = 11       # The epoch number to continue FROM (e.g. stopped at 10)
# ─────────────────────────────────────────────────────────────────────────
print(f'Training for: {LANGUAGE.upper()}')

Training for: FRENCH


## 1 · Imports & GPU

In [2]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import time

from src.config import (
    TRANS_ENG_VOCAB_SIZE, TRANS_TARGET_VOCAB_SIZE,
    TRANS_EMBEDDING_DIM, TRANS_UNITS,
    TRANS_MAX_TARGET_LEN,
)
from src.common import set_seed, configure_gpu
from src.translation.dataset import load_translation_artifacts
from src.translation.seq2seq import Seq2SeqTrainer

set_seed(42)
configure_gpu()
print(f'TensorFlow version: {tf.__version__}')

[INFO] No GPU detected — running on CPU.
TensorFlow version: 2.21.0


## 2 · Load Preprocessed Artifacts

In [3]:
data = load_translation_artifacts(LANGUAGE)

eng_train = data['eng_train']
tgt_train = data['tgt_train']
eng_test  = data['eng_test']
tgt_test  = data['tgt_test']
eng_tok   = data['eng_tokenizer']
tgt_tok   = data['tgt_tokenizer']

ENG_VOCAB = min(TRANS_ENG_VOCAB_SIZE,    len(eng_tok.word_index) + 1)
TGT_VOCAB = min(TRANS_TARGET_VOCAB_SIZE,  len(tgt_tok.word_index) + 1)

START_ID  = tgt_tok.word_index.get('<start>', 1)
END_ID    = tgt_tok.word_index.get('<end>',   2)

print(f'English vocab : {ENG_VOCAB:,}')
print(f'Target  vocab : {TGT_VOCAB:,}')
print(f'Train samples : {len(eng_train):,}')
print(f'Test  samples : {len(eng_test):,}')
print(f'<start> id    : {START_ID}  |  <end> id: {END_ID}')

[INFO] Tokenizer loaded  ←  c:\Users\himan\Documents\trial 2\models\translation\english_french_tokenizer.pkl
[INFO] Tokenizer loaded  ←  c:\Users\himan\Documents\trial 2\models\translation\french_tokenizer.pkl
English vocab : 15,000
Target  vocab : 15,000
Train samples : 191,677
Test  samples : 47,920
<start> id    : 2  |  <end> id: 3


## 3 · Build tf.data Datasets

In [4]:
train_dataset = Seq2SeqTrainer.make_tf_dataset(
    eng_train, tgt_train, batch_size=BATCH_SIZE, shuffle=True
)
steps_per_epoch = len(eng_train) // BATCH_SIZE

print(f'Steps per epoch: {steps_per_epoch}')

Steps per epoch: 2994


## 4 · Initialise Seq2Seq Trainer

In [5]:
trainer = Seq2SeqTrainer(
    eng_vocab_size = ENG_VOCAB,
    tgt_vocab_size = TGT_VOCAB,
    embedding_dim  = TRANS_EMBEDDING_DIM,
    units          = TRANS_UNITS,
    learning_rate  = LEARNING_RATE,
)

print('Encoder summary:')
trainer.encoder.summary()
print('\nDecoder summary:')
trainer.decoder.summary()

Encoder summary:


Model: "translation_encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ enc_embedding (Embedding)       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_emb_dropout (Dropout)       │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_bilstm (Bidirectional)      │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_fc_h (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_fc_c (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

...

Model: "translation_decoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dec_embedding (Embedding)       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bahdanau_attention              │ ?                      │   0 (unbuilt) │
│ (BahdanauAttention)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_gru (GRU)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_dropout (Dropout)           │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_output_fc (Dense)           │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# ── Resume from saved weights if requested ────────────────────────────────
if RESUME:
    from src.config import get_encoder_weights_path, get_decoder_weights_path
    import os

    enc_path = get_encoder_weights_path(LANGUAGE)
    dec_path = get_decoder_weights_path(LANGUAGE)

    if os.path.exists(enc_path) and os.path.exists(dec_path):
        trainer.load_weights(LANGUAGE)
        print(f"✅ Weights loaded. Resuming from epoch {START_EPOCH}.")
    else:
        print("⚠️  No saved weights found — starting from scratch.")
        RESUME = False
        START_EPOCH = 1

## 5 · Training Loop

In [ ]:
epoch_losses = []
best_loss    = float('inf')
patience_ctr = 0
PATIENCE     = 5

# Change range to start from START_EPOCH instead of 1
for epoch in range(START_EPOCH, EPOCHS + 1):
    start_time = time.time()
    epoch_loss = 0.0
    step       = 0

    for eng_batch, tgt_batch in train_dataset:
        step_loss   = trainer.train_step(
            tf.cast(eng_batch, tf.int32),
            tf.cast(tgt_batch, tf.int32),
            START_ID,
        )
        epoch_loss += step_loss.numpy()
        step       += 1

    avg_loss = epoch_loss / max(step, 1)
    elapsed  = time.time() - start_time
    epoch_losses.append(avg_loss)

    print(f'Epoch {epoch:>3}/{EPOCHS}  |  Loss: {avg_loss:.4f}  |  Time: {elapsed:.1f}s')

    if avg_loss < best_loss - 1e-4:
        best_loss    = avg_loss
        patience_ctr = 0
        trainer.save_weights(LANGUAGE)
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch}.')
            break

## 6 · Save Final Weights

In [ ]:
# Final save (in case the last epoch was the best)
trainer.save_weights(LANGUAGE)
print('Weights saved successfully.')

## 7 · Training Loss Plot

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(epoch_losses) + 1), epoch_losses,
         marker='o', linewidth=2, color='#3498DB', markersize=5)
plt.title(f'Seq2Seq Training Loss — {LANGUAGE.capitalize()}',
          fontsize=14, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss',  fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'training_loss_{LANGUAGE}.png', dpi=120)
plt.show()
print(f'Plot saved to training_loss_{LANGUAGE}.png')

## Training Complete

Weights saved to:
- `models/translation/encoder_{lang}.weights.h5`
- `models/translation/decoder_{lang}.weights.h5`